In [1]:
# ======================================================================
# 04_figures.ipynb
# FinGap-Governance | Figure generation for §7 (isomorphic skeleton)
# ----------------------------------------------------------------------
# Reads the three *_walkthrough.json files produced by notebooks 01-03
# and renders two publication figures in GREYSCALE (seaborn):
#   fig3a_traceability_matrix     — task x condition, traceable vs broken
#   fig3b_failure_by_condition    — condition x failure-type counts
# No captions embedded. Saved as BOTH .png and .pdf at dpi=600.
# Self-contained: no %run required.
# Set FINGAP_ROOT below if data/ is not in the current directory.
# ======================================================================

import os, json, glob
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

# ---- paths ------------------------------------------------------------
ROOT    = Path(os.environ.get("FINGAP_ROOT", ".")).resolve()
TABLES  = ROOT / "results" / "tables"
FIGURES = ROOT / "results" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

# ---- greyscale theme --------------------------------------------------
sns.set_theme(style="white", context="paper")
plt.rcParams.update({"font.family": "DejaVu Sans", "axes.edgecolor": "black"})

TASK_ORDER = ["advisory", "anomaly", "accounting"]
TASK_LABEL = {"advisory":   "Advisory\n(FCPA 17/18)",
              "anomaly":    "Anomaly\n(TFTIA STR/CTR)",
              "accounting": "Accounting\n(K-IFRS 1115)"}
COND_ORDER = ["direct", "vanilla_rag", "graph_rag"]
COND_LABEL = {"direct": "Direct", "vanilla_rag": "Vanilla RAG",
              "graph_rag": "GraphRAG"}
FAILS = ["boundary_misjudgment", "provision_omission", "misattribution",
         "logic_disconnection", "hallucination", "(none)"]

# ---- load walkthrough results -----------------------------------------
data = {}
for fp in glob.glob(str(TABLES / "*_walkthrough.json")):
    d = json.load(open(fp, encoding="utf-8"))
    data[d["task"]] = {v["condition"]: v for v in d["verdicts"]}
assert data, f"no *_walkthrough.json in {TABLES} — run notebooks 01-03 first"

def savefig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIGURES / f"{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)

# ---- Figure 3a: traceability matrix (task x condition) ----------------
mat = np.array([[1 if data[t][c]["traceable"] else 0 for c in COND_ORDER]
                for t in TASK_ORDER])
fig, ax = plt.subplots(figsize=(5.0, 3.2))
sns.heatmap(mat, cmap="Greys", vmin=0, vmax=1, cbar=False,
            linewidths=1.2, linecolor="black", square=True,
            xticklabels=[COND_LABEL[c] for c in COND_ORDER],
            yticklabels=[TASK_LABEL[t] for t in TASK_ORDER], ax=ax)
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        ax.text(j + 0.5, i + 0.5,
                "traceable" if mat[i, j] else "broken",
                ha="center", va="center",
                color="white" if mat[i, j] else "black", fontsize=8)
ax.set_xlabel(""); ax.set_ylabel("")
plt.setp(ax.get_yticklabels(), rotation=0)
savefig(fig, "fig3a_traceability_matrix")

# ---- Figure 3b: failure-type by condition -----------------------------
fmat = np.zeros((len(COND_ORDER), len(FAILS)), dtype=int)
for t in TASK_ORDER:
    for ci, c in enumerate(COND_ORDER):
        f = data[t][c]["failure"] or "(none)"
        fmat[ci, FAILS.index(f)] += 1
fig, ax = plt.subplots(figsize=(6.2, 3.0))
sns.heatmap(fmat, cmap="Greys", cbar=True,
            cbar_kws={"label": "count (of 3 tasks)"},
            linewidths=1.0, linecolor="black",
            xticklabels=[f.replace("_", "\n") for f in FAILS],
            yticklabels=[COND_LABEL[c] for c in COND_ORDER],
            annot=True, fmt="d", annot_kws={"fontsize": 9}, ax=ax)
ax.set_xlabel(""); ax.set_ylabel("")
plt.setp(ax.get_xticklabels(), rotation=0, fontsize=7)
plt.setp(ax.get_yticklabels(), rotation=0)
savefig(fig, "fig3b_failure_by_condition")

print("generated (png + pdf @ dpi 600):")
for p in sorted(FIGURES.glob("fig3*")):
    print("  ", p.name)

generated (png + pdf @ dpi 600):
   fig3a_traceability_matrix.pdf
   fig3a_traceability_matrix.png
   fig3b_failure_by_condition.pdf
   fig3b_failure_by_condition.png
